# 👨‍💻 Portfólio — Cytussolo | Analista de Dados Espaciais

**Manaus, AM — Brasil**

Este notebook executa os 3 projetos completos do portfólio:

| # | Projeto | Output |
|---|---------|--------|
| 1 | 🌍 Análise Geoespacial | `mapa_brasil.png` |
| 2 | 🧊 Modelagem 3D de Terreno | `terrain.png` + `terrain_model.obj` |
| 3 | 📊 Dashboard de Dados | `dashboard.png` + `dados_vendas.csv` |

---
> ▶️ **Execute célula por célula** (Shift+Enter) ou vá em **Runtime → Run all**

## ⚙️ PASSO 1 — Instalar dependências
Execute esta célula primeiro. Leva cerca de 1-2 minutos.

In [ ]:
# Instalar tudo de uma vez
import subprocess, sys

print('📦 Instalando dependências...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'geopandas', 'matplotlib', 'numpy', 'pandas',
    'shapely', 'fiona', 'pyproj'
], check=True)
print('✅ Tudo instalado!')

---
## 🌍 PROJETO 1 — Análise Geoespacial com Python

Gera dois mapas:
- **Esquerda:** Mapa de população da América do Sul (heatmap)
- **Direita:** Brasil em destaque (verde)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('=' * 60)
print('🌍 ANÁLISE GEOESPACIAL COM PYTHON')
print('=' * 60)

# Carregando dados
print('\n📍 Carregando dados geográficos...')
world         = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
brazil        = world[world['name'] == 'Brazil']
south_america = world[world['continent'] == 'South America']
print(f'✅ {len(south_america)} países carregados')

# Figura
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0a0e1a')

# --- Subplot 1: Mapa de população ---
south_america.plot(
    column='pop_est', ax=axes[0], legend=True,
    cmap='YlOrRd', edgecolor='white', linewidth=0.5
)
axes[0].set_facecolor('#0d1a30')
axes[0].set_title('🌍 População — América do Sul',
    color='white', fontsize=14, fontweight='bold', pad=12)
axes[0].set_axis_off()

# --- Subplot 2: Brasil em destaque ---
south_america.plot(ax=axes[1], color='#1a2540', edgecolor='#334155')
brazil.plot(ax=axes[1], color='#00ff9d', alpha=0.8, edgecolor='white', linewidth=1.5)
axes[1].set_facecolor('#0d1a30')
axes[1].set_title('🇧🇷 Brasil em Destaque',
    color='white', fontsize=14, fontweight='bold', pad=12)
axes[1].set_axis_off()

plt.tight_layout()
plt.savefig('mapa_brasil.png', dpi=200, bbox_inches='tight', facecolor='#0a0e1a')
print('\n✅ mapa_brasil.png salvo!')
plt.show()

# Estatísticas
print('\n📈 POPULAÇÃO POR PAÍS:')
print('-' * 45)
tabela = south_america[['name','pop_est']].sort_values('pop_est', ascending=False)
tabela.columns = ['País', 'População']
tabela['População'] = tabela['População'].apply(lambda x: f'{x:,.0f}')
tabela = tabela.reset_index(drop=True)
tabela.index += 1
print(tabela.to_string())

---
## 🧊 PROJETO 2 — Modelagem 3D de Terreno

Gera:
- **Visualização 3D** do terreno com colormap de altitude
- **Heatmap 2D** (visão de cima)
- **Arquivo .OBJ** para abrir no Blender

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print('=' * 60)
print('🧊 GERAÇÃO DE TERRENO 3D')
print('=' * 60)

# Parâmetros
SIZE, SCALE = 100, 10

# Grade
x = np.linspace(0, 10, SIZE)
y = np.linspace(0, 10, SIZE)
X, Y = np.meshgrid(x, y)

# Altura (simula Perlin Noise com funções trigonométricas)
Z = SCALE * (
    np.sin(X/2)   * np.cos(Y/2) +
    0.5 * np.sin(X)   * np.cos(Y/3) +
    0.3 * np.sin(X/3) * np.cos(Y)
)
print(f'✅ Grade {SIZE}x{SIZE} gerada ({SIZE*SIZE:,} vértices)')

# Figura
fig = plt.figure(figsize=(16, 6), facecolor='#0a0e1a')

# --- 3D Surface ---
ax1 = fig.add_subplot(121, projection='3d')
ax1.set_facecolor('#0d1a30')
surf = ax1.plot_surface(X, Y, Z, cmap='terrain', alpha=0.92, linewidth=0)
ax1.set_xlabel('X', color='white')
ax1.set_ylabel('Y', color='white')
ax1.set_zlabel('Altitude', color='white')
ax1.tick_params(colors='white')
ax1.set_title('🏔️ Terreno 3D', color='white', fontweight='bold', pad=10)
fig.colorbar(surf, ax=ax1, label='Altitude', shrink=0.5)

# --- Heatmap 2D ---
ax2 = fig.add_subplot(122)
ax2.set_facecolor('#0d1a30')
c = ax2.contourf(X, Y, Z, levels=25, cmap='terrain')
ax2.contour(X, Y, Z, levels=10, colors='white', alpha=0.15, linewidths=0.5)
ax2.set_xlabel('X', color='white')
ax2.set_ylabel('Y', color='white')
ax2.tick_params(colors='white')
ax2.set_title('🗺️ Heatmap de Altitude', color='white', fontweight='bold')
fig.colorbar(c, ax=ax2, label='Altitude')

plt.tight_layout()
plt.savefig('terrain.png', dpi=200, bbox_inches='tight', facecolor='#0a0e1a')
print('✅ terrain.png salvo!')
plt.show()

# --- Exportar OBJ para Blender ---
print('\n💾 Exportando terrain_model.obj...')
with open('terrain_model.obj', 'w') as f:
    f.write('# Terrain Model — Cytussolo | Manaus, AM\n')
    f.write(f'# Grid: {SIZE}x{SIZE}\n\n')
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            f.write(f'v {X[i,j]:.4f} {Y[i,j]:.4f} {Z[i,j]:.4f}\n')
    f.write('\n')
    for i in range(X.shape[0] - 1):
        for j in range(X.shape[1] - 1):
            v1 = i * X.shape[1] + j + 1
            v2, v3, v4 = v1+1, v1+X.shape[1], v1+X.shape[1]+1
            f.write(f'f {v1} {v2} {v3}\nf {v2} {v4} {v3}\n')

print('✅ terrain_model.obj salvo! (Abrir no Blender)')
print(f'\n📊 Altitude  min: {Z.min():.2f} | max: {Z.max():.2f} | média: {Z.mean():.2f}')

---
## 📊 PROJETO 3 — Dashboard de Dados

Gera:
- **500 registros** de vendas simulados
- **CSV** pronto para importar no Power BI
- **4 gráficos** em um dashboard: por categoria, por região, evolução mensal e lucro vs custo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

print('=' * 60)
print('📊 GERAÇÃO DE DADOS — DASHBOARD')
print('=' * 60)

np.random.seed(42)
N = 500
categorias = ['Eletrônicos', 'Roupas', 'Alimentos', 'Móveis', 'Livros']
regioes    = ['Norte', 'Sul', 'Leste', 'Oeste', 'Centro']
start      = datetime(2023, 1, 1)

df = pd.DataFrame({
    'data':      [start + timedelta(days=int(np.random.randint(0, 365))) for _ in range(N)],
    'categoria': np.random.choice(categorias, N),
    'regiao':    np.random.choice(regioes, N),
    'vendas':    np.random.uniform(50, 5000, N).round(2),
    'unidades':  np.random.randint(1, 50, N),
    'custo':     np.random.uniform(20, 3000, N).round(2),
})
df['lucro'] = (df['vendas'] - df['custo']).round(2)
df['mes']   = df['data'].apply(lambda d: d.strftime('%Y-%m'))

df.to_csv('dados_vendas.csv', index=False)
print(f'✅ dados_vendas.csv salvo: {N} registros')
print(f'   Total vendas: R$ {df["vendas"].sum():,.2f}')
print(f'   Total lucro:  R$ {df["lucro"].sum():,.2f}')

# Dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='#0a0e1a')
colors = ['#00e5ff', '#00ff9d', '#7c3aed', '#f59e0b', '#ef4444']

# 1 — Vendas por categoria
cat = df.groupby('categoria')['vendas'].sum().sort_values(ascending=True)
bars = axes[0,0].barh(cat.index, cat.values, color=colors)
axes[0,0].set_title('💰 Vendas por Categoria', color='white', fontweight='bold')
axes[0,0].set_facecolor('#111827')
axes[0,0].tick_params(colors='white')
for sp in axes[0,0].spines.values(): sp.set_color('#1a2235')
for bar, val in zip(bars, cat.values):
    axes[0,0].text(val + 500, bar.get_y() + bar.get_height()/2,
        f'R${val/1000:.0f}k', va='center', color='white', fontsize=9)

# 2 — Vendas por região
reg = df.groupby('regiao')['vendas'].sum()
axes[0,1].pie(reg, labels=reg.index, colors=colors,
    autopct='%1.1f%%', textprops={'color':'white'}, startangle=90)
axes[0,1].set_title('🗺️ Vendas por Região', color='white', fontweight='bold')
axes[0,1].set_facecolor('#111827')

# 3 — Evolução mensal
mensal = df.groupby('mes')['vendas'].sum().sort_index()
x_idx = range(len(mensal))
axes[1,0].plot(x_idx, mensal.values, color='#00e5ff', linewidth=2.5, marker='o', markersize=5)
axes[1,0].fill_between(x_idx, mensal.values, alpha=0.12, color='#00e5ff')
axes[1,0].set_xticks(x_idx)
axes[1,0].set_xticklabels(mensal.index, rotation=45, ha='right', color='white', fontsize=8)
axes[1,0].set_title('📈 Evolução Mensal', color='white', fontweight='bold')
axes[1,0].set_facecolor('#111827')
axes[1,0].tick_params(colors='white')
for sp in axes[1,0].spines.values(): sp.set_color('#1a2235')

# 4 — Lucro vs Custo
lc = df.groupby('categoria')[['lucro','custo']].sum()
x  = np.arange(len(lc))
axes[1,1].bar(x - 0.2, lc['lucro'], 0.38, color='#00ff9d', label='Lucro')
axes[1,1].bar(x + 0.2, lc['custo'], 0.38, color='#ef4444', label='Custo')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(lc.index, rotation=15, color='white', fontsize=9)
axes[1,1].set_title('💹 Lucro vs Custo por Categoria', color='white', fontweight='bold')
axes[1,1].set_facecolor('#111827')
axes[1,1].tick_params(colors='white')
axes[1,1].legend(facecolor='#1a2235', labelcolor='white')
for sp in axes[1,1].spines.values(): sp.set_color('#1a2235')

fig.suptitle('📊 Dashboard de Vendas — Cytussolo', color='white',
    fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('dashboard.png', dpi=200, bbox_inches='tight', facecolor='#0a0e1a')
print('\n✅ dashboard.png salvo!')
plt.show()

---
## 📥 PASSO FINAL — Baixar todos os arquivos gerados

In [ ]:
from google.colab import files
import os

outputs = [
    'mapa_brasil.png',
    'terrain.png',
    'terrain_model.obj',
    'dashboard.png',
    'dados_vendas.csv',
]

print('📦 Baixando arquivos gerados...\n')
for f in outputs:
    if os.path.exists(f):
        print(f'  ⬇️  {f}')
        files.download(f)
    else:
        print(f'  ⚠️  {f} não encontrado — execute a célula do projeto primeiro')

print('\n✅ Concluído! Confira a pasta de downloads do seu navegador.')

---
## ✅ Resumo do que foi gerado

| Arquivo | Descrição | Usar com |
|---------|-----------|----------|
| `mapa_brasil.png` | Mapa da América do Sul | GitHub / Portfólio |
| `terrain.png` | Visualização 3D do terreno | GitHub / Portfólio |
| `terrain_model.obj` | Modelo 3D do terreno | Blender |
| `dashboard.png` | Dashboard de vendas | GitHub / Portfólio |
| `dados_vendas.csv` | Dados de vendas | Power BI |

---
*Cytussolo · Analista de Dados Espaciais · Manaus, AM*